<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Wrangling Lab**


Estimated time needed: **45** minutes


In this lab, you will perform data wrangling tasks to prepare raw data for analysis. Data wrangling involves cleaning, transforming, and organizing data into a structured format suitable for analysis. This lab focuses on tasks like identifying inconsistencies, encoding categorical variables, and feature transformation.


## Objectives


After completing this lab, you will be able to:


- Identify and remove inconsistent data entries.

- Encode categorical variables for analysis.

- Handle missing values using multiple imputation strategies.

- Apply feature scaling and transformation techniques.


#### Intsall the required libraries


In [ ]:
!pip install pandas
!pip install matplotlib

## Tasks


#### Step 1: Import the necessary module.


### 1. Load the Dataset


<h5>1.1 Import necessary libraries and load the dataset.</h5>


Ensure the dataset is loaded correctly by displaying the first few rows.


In [ ]:
# Import necessary libraries
import pandas as pd

# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())


#### 2. Explore the Dataset


<h5>2.1 Summarize the dataset by displaying the column data types, counts, and missing values.</h5>


In [ ]:
# 2.1 Column data types, non-null counts and missing values
df.info()
print("\nMissing values per column:")
print(df.isnull().sum().sort_values(ascending=False))

<h5>2.2 Generate basic statistics for numerical columns.</h5>


In [ ]:
# 2.2 Basic statistics for numerical columns
df.describe()

### 3. Identifying and Removing Inconsistencies


<h5>3.1 Identify inconsistent or irrelevant entries in specific columns (e.g., Country).</h5>


In [ ]:
# 3.1 Look for inconsistent or irrelevant entries in Country
print("Number of unique countries:", df['Country'].nunique())
print(df['Country'].value_counts().to_string())

# Entries with extra spaces or unusually long official names are typical inconsistencies
countries = pd.Series(df['Country'].dropna().unique())
print("\nEntries with leading/trailing spaces:", countries[countries != countries.str.strip()].tolist())
print("Long official-style names:", countries[countries.str.len() > 30].tolist())

<h5>3.2 Standardize entries in columns like Country or EdLevel by mapping inconsistent values to a consistent format.</h5>


In [ ]:
# 3.2 Standardize Country and EdLevel values
df['Country'] = df['Country'].str.strip()
country_map = {
    'United States of America': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK',
    'Russian Federation': 'Russia',
    'Iran, Islamic Republic of...': 'Iran',
    'Republic of Korea': 'South Korea',
    'Viet Nam': 'Vietnam',
    'Venezuela, Bolivarian Republic of...': 'Venezuela',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'The former Yugoslav Republic of Macedonia': 'North Macedonia',
}
df['Country'] = df['Country'].replace(country_map)

edlevel_map = {
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)": "Bachelor's",
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master's",
    "Some college/university study without earning a degree": "Some College",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary School",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "Professional Degree",
    "Associate degree (A.A., A.S., etc.)": "Associate Degree",
    "Primary/elementary school": "Primary School",
    "Something else": "Other",
}
df['EdLevel'] = df['EdLevel'].replace(edlevel_map)

print(df['Country'].value_counts().head(10))
print(df['EdLevel'].value_counts())

### 4. Encoding Categorical Variables


<h5>4.1 Encode the Employment column using one-hot encoding.</h5>


In [ ]:
# 4.1 One-hot encode Employment
# Employment can hold several answers separated by ';', so split them into one column per status
employment_dummies = df['Employment'].str.get_dummies(sep=';').add_prefix('Employment_')
df = pd.concat([df, employment_dummies], axis=1)

print("New one-hot columns:", employment_dummies.shape[1])
employment_dummies.head()

### 5. Handling Missing Values


<h5>5.1 Identify columns with the highest number of missing values.</h5>


In [ ]:
# 5.1 Columns with the highest number of missing values
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'Missing': missing, 'Percent': missing_pct}).head(10)

<h5>5.2 Impute missing values in numerical columns (e.g., `ConvertedCompYearly`) with the mean or median.</h5>


In [ ]:
# 5.2 Impute ConvertedCompYearly with the median (compensation is skewed, so median beats mean)
median_comp = df['ConvertedCompYearly'].median()
df['ConvertedCompYearly'] = df['ConvertedCompYearly'].fillna(median_comp)
print("Median used:", median_comp)
print("Missing ConvertedCompYearly after imputation:", df['ConvertedCompYearly'].isnull().sum())

<h5>5.3 Impute missing values in categorical columns (e.g., `RemoteWork`) with the most frequent value.</h5>


In [ ]:
# 5.3 Impute RemoteWork with the most frequent value
most_frequent_remote = df['RemoteWork'].mode()[0]
df['RemoteWork'] = df['RemoteWork'].fillna(most_frequent_remote)
print("Most frequent RemoteWork:", most_frequent_remote)
print("Missing RemoteWork after imputation:", df['RemoteWork'].isnull().sum())

### 6. Feature Scaling and Transformation


<h5>6.1 Apply Min-Max Scaling to normalize the `ConvertedCompYearly` column.</h5>


In [ ]:
# 6.1 Min-Max scaling of ConvertedCompYearly
comp_min = df['ConvertedCompYearly'].min()
comp_max = df['ConvertedCompYearly'].max()
df['ConvertedCompYearly_MinMax'] = (df['ConvertedCompYearly'] - comp_min) / (comp_max - comp_min)
df[['ConvertedCompYearly', 'ConvertedCompYearly_MinMax']].describe()

<h5>6.2 Log-transform the ConvertedCompYearly column to reduce skewness.</h5>


In [ ]:
# 6.2 Log-transform ConvertedCompYearly to reduce skewness
import numpy as np
import matplotlib.pyplot as plt

df['ConvertedCompYearly_Log'] = np.log1p(df['ConvertedCompYearly'])   # log(1 + x) handles zeros safely

print("Skewness before:", round(df['ConvertedCompYearly'].skew(), 2))
print("Skewness after :", round(df['ConvertedCompYearly_Log'].skew(), 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['ConvertedCompYearly'], bins=50, color='steelblue')
axes[0].set_title('Original ConvertedCompYearly')
axes[1].hist(df['ConvertedCompYearly_Log'], bins=50, color='seagreen')
axes[1].set_title('Log-transformed ConvertedCompYearly')
plt.tight_layout()
plt.show()

### 7. Feature Engineering


<h5>7.1 Create a new column `ExperienceLevel` based on the `YearsCodePro` column:</h5>


In [ ]:
# 7.1 ExperienceLevel from YearsCodePro
# YearsCodePro is text: convert the two word answers to numbers first
years = df['YearsCodePro'].replace({'Less than 1 year': '0', 'More than 50 years': '51'})
df['YearsCodePro_Num'] = pd.to_numeric(years, errors='coerce')

df['ExperienceLevel'] = pd.cut(
    df['YearsCodePro_Num'],
    bins=[-1, 2, 5, 10, 100],
    labels=['Beginner (0-2 yrs)', 'Intermediate (3-5 yrs)', 'Experienced (6-10 yrs)', 'Expert (10+ yrs)']
)

print(df['ExperienceLevel'].value_counts())
df[['YearsCodePro', 'ExperienceLevel']].head(10)

### Summary


In this lab, you:

- Explored the dataset to identify inconsistencies and missing values.

- Encoded categorical variables for analysis.

- Handled missing values using imputation techniques.

- Normalized and transformed numerical data to prepare it for analysis.

- Engineered a new feature to enhance data interpretation.


Copyright © IBM Corporation. All rights reserved.
